#   分析和可视化人口普查租金数据



In [16]:
import pandas as pd, numpy as np
import statsmodels.api as sm, matplotlib.font_manager as fm
import matplotlib.pyplot as plt, matplotlib.cm as cm
from mpl_toolkits.basemap import Basemap
from geopandas import GeoDataFrame
from shapely.geometry import Point

%matplotlib inline

D:\Anaconda\envs\rent-data-analysis\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [44]:
# 读取 2014 年的租金数据
rent14 = pd.read_csv('../data/rent_latlong.csv')
# 对数据列重命名
rent14 = rent14.rename(columns={'geo_id2':'geo_id_14'})    #2014 年的地理编码
# 剔除无关的数据列
# 2014 年地理编码（唯一标识）、大都市区名称、城市 + 州简称、合同租金中位数、纬度、经度
rent14 = rent14[['geo_id_14', 'msa_name', 'city_state', 'median_contract_rent', 'latitude', 'longitude']]
rent14.head()

,geo_id_14,msa_name,city_state,median_contract_rent,latitude,longitude
0,10140,"Aberdeen, WA Micro Area","Aberdeen, WA",626,46.975371,-123.815722
1,10180,"Abilene, TX Metro Area","Abilene, TX",613,32.448736,-99.733144
2,10300,"Adrian, MI Micro Area","Adrian, MI",556,41.897547,-84.037166
3,10420,"Akron, OH Metro Area","Akron, OH",627,41.081445,-81.519005
4,10460,"Alamogordo, NM Micro Area","Alamogordo, NM",700,32.899532,-105.960265


In [45]:
# 读取 2010 年的租金数据，并规范化列名，剔除无关的数据列
rent10 = pd.read_csv('../data/ACS_10_1YR_B25058_metro_micro/ACS_10_1YR_B25058.csv')
rent10 = rent10.rename(columns={'GEO.id':'geo_id', 
                            'GEO.id2':'geo_id_10', 
                            'GEO.display-label':'msa_name', 
                            'HD01_VD01':'median_rent_10',
                            'HD02_VD01':'margin_error'})
# 2010 年的地理编码、2010 年该地区的合同租金中位数
rent10 = rent10[['geo_id_10', 'median_rent_10']]
rent10.head()

,geo_id_10,median_rent_10
0,10140,562
1,10180,554
2,10300,537
3,10420,579
4,10500,477


## 合并2014年和2010年的ACS租金数据

一些人口普查统计区的名称和代码在2010年和2014年之间发生了变化。为那些发生变化的区域建立一个字典，将2014年的代码映射到2010年的代码。其他的一些是全新的，还有一些在2014年被删除的只需忽略它们，因为除非它们在两年都有数据，否则无法比较。

In [39]:
# create a dict to map 2014 codes to 2010 codes for those that changed
# 创建一个字典，将2014年的代码映射到已更改的2010年的代码
# 因为部分地区的地理编码在 2010-2014 年间发生了变更（比如 bloomington il 的编码从 2010 年的 14060 变成了 2014 年的 14010），需要通过这个字典记录 “变动的编码对应关系”，方便后续匹配两年的数据。
codes_14_10 = {  #2014_code:2010_code
    14010:14060, #bloomington il
    15680:30500, #lexington park md
    16060:32060, #marion il
    17200:30100, #lebanon nh
    25840:37820, #pendleton
    26090:26100, #holland
    29200:29140, #lafayette
    38240:43860, #pinehurst
    41400:20620, #salem oh
    49220:32270, #wisconsin rapids
    31080:31100, #los angeles
    42200:42060, #santa barbara
    46520:26180, #honolulu
    48260:44600} #steubenville

# 创建一个新列以包含 2010 年的地理编码，方便后面将 rent14 和 rent10 合并起来
rent14['geo_id_10'] = rent14['geo_id_14'].map(lambda x: codes_14_10[x] if x in codes_14_10.keys() else x)

In [38]:
# 现在在 2010 年的地理编码上合并 2014 年的租金数据和 2010 年的租金数据
rent = pd.merge(rent14, rent10, on='geo_id_10')  #两表都有共同列‘geo_id_10’
rent.head()

,geo_id_14,msa_name,city_state,median_contract_rent,latitude,longitude,geo_id_10,median_rent_10
0,10140,"Aberdeen, WA Micro Area","Aberdeen, WA",626,46.975371,-123.815722,10140,562
1,10180,"Abilene, TX Metro Area","Abilene, TX",613,32.448736,-99.733144,10180,554
2,10300,"Adrian, MI Micro Area","Adrian, MI",556,41.897547,-84.037166,10300,537
3,10420,"Akron, OH Metro Area","Akron, OH",627,41.081445,-81.519005,10420,579
4,10500,"Albany, GA Metro Area","Albany, GA",480,31.578507,-84.155741,10500,477


In [43]:
# 计算 2010-2014 年各地区租金中位数的变化百分比，然后按涨幅从低到高排序，筛选出涨幅最低（甚至下跌）的前 5 个地区并展示其名称和涨幅百分比，用于分析租金变化的极端情况
# 租金变化百分比 = (2014 年租金 / 2010 年租金 - 1) × 100
rent['rent_change_pct'] = (rent['median_contract_rent'] / rent['median_rent_10'] - 1) * 100
rent.sort_values(by='rent_change_pct')[['msa_name', 'rent_change_pct']].head()

,msa_name,rent_change_pct
414,"Seneca, SC Micro Area",-14.417745
413,"Sebring, FL Metro Area",-11.992945
69,"Carson City, NV Metro Area",-9.015257
464,"Valdosta, GA Metro Area",-7.992895
442,"Talladega-Sylacauga, AL Micro Area",-7.049608


## 导入人口数据并与租金数据合并

包含 2014 年的人口预估数和 2010 年的人口普查数值

In [42]:
# 读取 2014 年人口统计数据的 CSV 文件，规范化列名，剔除无关列数据
pops = pd.read_csv('../data/PEP_2014_PEPANNRES/PEP_2014_PEPANNRES.csv', encoding='utf-8')
pops = pops.rename(columns={'GEO.id2':'geo_id_14',           #2014 年的地理编码
                              'rescen42010':'pop_10',        #2010 年人口普查的常住人口数
                              'respop72014':'pop_est_14'})   #2014 年人口预估数
pops = pops[['geo_id_14', 'pop_10', 'pop_est_14']]
pops.head()

,geo_id_14,pop_10,pop_est_14
0,10100,40602,42391
1,10140,72797,70818
2,10180,165252,168592
3,10220,37492,38005
4,10300,99892,99047


In [51]:
# 将租金数据集与人口数据集合并
df = pd.merge(rent, pops, on='geo_id_14')
df.head()

,geo_id_14,msa_name,city_state,median_contract_rent,latitude,longitude,geo_id_10,median_rent_10,rent_change_pct,pop_10,pop_est_14
0,10140,"Aberdeen, WA Micro Area","Aberdeen, WA",626,46.975371,-123.815722,10140,562,11.387900,72797,70818
1,10180,"Abilene, TX Metro Area","Abilene, TX",613,32.448736,-99.733144,10180,554,10.649819,165252,168592
2,10300,"Adrian, MI Micro Area","Adrian, MI",556,41.897547,-84.037166,10300,537,3.538175,99892,99047
3,10420,"Akron, OH Metro Area","Akron, OH",627,41.081445,-81.519005,10420,579,8.290155,703200,703825
4,10500,"Albany, GA Metro Area","Albany, GA",480,31.578507,-84.155741,10500,477,0.628931,157308,154925


In [50]:
# 计算 2010-2014 年各地区人口变化的百分比，按涨幅从低到高排序后，筛选出人口跌幅最大（或涨幅最低）的前 5 个地区，仅展示地区名称和人口变化百分比，用于分析人口变化的极端情况
# 人口变化百分比 = (2014 年预估人口 / 2010 年普查人口 - 1) × 100
df['pop_change_pct'] = (df['pop_est_14'] / df['pop_10'] - 1) * 100
df.sort_values(by='pop_change_pct')[['msa_name', 'pop_change_pct']].head()
# df.head()

,msa_name,pop_change_pct
348,"Pine Bluff, AR Metro Area",-5.527738
142,"Farmington, NM Metro Area",-4.812986
377,"Roanoke Rapids, NC Micro Area",-4.371663
214,"Johnstown, PA Metro Area",-4.139088
277,"Martinsville, VA Micro Area",-3.207203
